# Sprint 4 — Modelos avanzados y ensambles

Objetivo de este notebook: ejecutar y guardar resultados de modelos avanzados y ensambles.  
La comparación final contra baselines y modelos tuneados se realizará en el notebook 13.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import *
from src.preprocessing import split_X_y, build_preprocessor
from src.models import evaluate_cv, summarize_cv_scores

import joblib

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 120)

print("Proyecto:", PROJECT_ROOT)

N_JOBS = 1


Proyecto: /Users/alexandralozano/dp261-g1


In [2]:
# 1. Cargar muestra estratificada y modelos tuneados disponibles

train_sample = pd.read_csv(PROCESSED_DIR / "train_sample.csv")
X_sample, y_sample = split_X_y(train_sample)

tuned_paths = list(MODELS_DIR.glob("tuned_*.pkl"))
print("Modelos tuneados encontrados:", [p.name for p in tuned_paths])

tuned = {
    p.stem.replace("tuned_", ""): joblib.load(p)
    for p in tuned_paths
}

print("Modelos cargados:", list(tuned.keys()))
print("X_sample:", X_sample.shape)
print("Distribución target:")
display(y_sample.value_counts(normalize=True).rename("proporcion"))


Modelos tuneados encontrados: ['tuned_DecisionTree.pkl', 'tuned_LinearSVM.pkl', 'tuned_LogisticRegression.pkl']
Modelos cargados: ['DecisionTree', 'LinearSVM', 'LogisticRegression']
X_sample: (20000, 40)
Distribución target:


IsBadBuy
0    0.877
1    0.123
Name: proporcion, dtype: float64

In [3]:
# 2. Evaluar modelos tuneados cargados
tuned_rows = []

for name, model in tuned.items():
    scores = evaluate_cv(model, X_sample, y_sample)
    tuned_rows.append(summarize_cv_scores(name, scores))

tuned_loaded_results = (
    pd.DataFrame(tuned_rows)
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

cols_to_show = [
    "model",
    "recall_cv_mean",
    "f2_cv_mean",
    "precision_cv_mean",
    "f1_cv_mean",
    "f05_cv_mean",
    "roc_auc_cv_mean",
    "average_precision_cv_mean",
    "fit_time_mean",
]

cols_to_show = [col for col in cols_to_show if col in tuned_loaded_results.columns]

display(tuned_loaded_results[cols_to_show])

tuned_loaded_results.to_csv(
    REPORTS_DIR / "tuned_loaded_results.csv",
    index=False
)


,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,roc_auc_cv_mean,average_precision_cv_mean,fit_time_mean
0,DecisionTree,0.613821,0.456661,0.228596,0.331660,0.260947,0.718980,0.421301,0.485702
1,LogisticRegression,0.610976,0.483195,0.263126,0.367824,0.296933,0.754036,0.436023,0.378450
2,LinearSVM,0.603252,0.481969,0.267182,0.370319,0.300677,0.754027,0.430728,0.423806


In [4]:
%pip install xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.


In [5]:
# 3. Modelos avanzados de Gradient Boosting: XGBoost y LightGBM
advanced_rows = []
advanced_models = {}

scale_pos_weight = (y_sample == 0).sum() / max((y_sample == 1).sum(), 1)

# XGBoost
from xgboost import XGBClassifier

advanced_models["XGBoost"] = Pipeline(steps=[
    ("preprocessor", build_preprocessor(X_sample, mode="ordinal")),
    ("model", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    ))
])


# LightGBM
from lightgbm import LGBMClassifier

advanced_models["LightGBM"] = Pipeline(steps=[
    ("preprocessor", build_preprocessor(X_sample, mode="ordinal")),
    ("model", LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        verbose=1
    ))
])


for name, model in advanced_models.items():
    print(f"Evaluando {name}...")
    scores = evaluate_cv(model, X_sample, y_sample)
    advanced_rows.append(summarize_cv_scores(name, scores))

advanced_boosting_results = (
    pd.DataFrame(advanced_rows)
    .assign(source="advanced_boosting")
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
    if advanced_rows else pd.DataFrame()
)

advanced_boosting_results.to_csv(
    REPORTS_DIR / "advanced_boosting_results.csv",
    index=False
)

display(advanced_boosting_results)

Evaluando XGBoost...
Evaluando LightGBM...
[LightGBM] [Info] Number of positive: 1968, number of negative: 14032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001454 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5230
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1968, number of negative: 14032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001278 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5225
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1968, number of negative: 14032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001276 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5228
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

[LightGBM] [Info] Number of positive: 1968, number of negative: 14032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001249 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5220
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1968, number of negative: 14032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001220 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5230
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,fit_time_mean,accuracy_cv_mean,accuracy_cv_std,accuracy_train_mean,accuracy_gap,precision_cv_mean,precision_cv_std,precision_train_mean,precision_gap,recall_cv_mean,recall_cv_std,recall_train_mean,recall_gap,f1_cv_mean,f1_cv_std,f1_train_mean,f1_gap,f05_cv_mean,f05_cv_std,f05_train_mean,f05_gap,f2_cv_mean,f2_cv_std,f2_train_mean,f2_gap,roc_auc_cv_mean,roc_auc_cv_std,roc_auc_train_mean,roc_auc_gap,average_precision_cv_mean,average_precision_cv_std,average_precision_train_mean,average_precision_gap,source
0,XGBoost,0.309506,0.76990,0.002228,0.799350,0.029450,0.285934,0.008261,0.346409,0.060475,0.582520,0.032383,0.710366,0.127846,0.383508,0.014382,0.465651,0.082143,0.318324,0.010090,0.385936,0.067612,0.482343,0.022416,0.586935,0.104592,0.766292,0.014109,0.855921,0.089629,0.477231,0.027078,0.582993,0.105762,advanced_boosting
1,LightGBM,0.404001,0.79745,0.003750,0.868275,0.070825,0.311559,0.014142,0.480401,0.168842,0.536585,0.040630,0.863211,0.326626,0.394115,0.022288,0.617217,0.223102,0.340039,0.016737,0.527135,0.187096,0.468746,0.031244,0.744489,0.275743,0.766080,0.014468,0.945850,0.179770,0.479273,0.028357,0.750400,0.271127,advanced_boosting


In [6]:
# 4. BaggingClassifier sobre DecisionTree

bagging_rows = []

bagging_tree = Pipeline(steps=[
    ("preprocessor", build_preprocessor(
        X_sample,
        mode="tree_ohe",
        min_frequency=0.01
    )),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        n_estimators=50,
        max_samples=0.8,
        max_features=0.8,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    ))
])

scores = evaluate_cv(bagging_tree, X_sample, y_sample)
bagging_rows.append(summarize_cv_scores("Bagging_DecisionTree", scores))

bagging_results = (
    pd.DataFrame(bagging_rows)
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(bagging_results[cols_to_show])

bagging_results.to_csv(
    REPORTS_DIR / "bagging_results.csv",
    index=False
)


,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,roc_auc_cv_mean,average_precision_cv_mean,fit_time_mean
0,Bagging_DecisionTree,0.53252,0.465843,0.31067,0.392293,0.33886,0.756702,0.468173,6.370873


In [7]:
# 5. Ensambles con mejores modelos tuneados: Voting y Stacking
ensemble_rows = []

estimators = list(tuned.items())[:3]

ensembles = {
    "Voting_soft": VotingClassifier(
        estimators=estimators,
        voting="soft"
    ),
    "Stacking_lr": StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000),
        cv=3
    )
}

for name, model in ensembles.items():
    scores = evaluate_cv(model, X_sample, y_sample)
    ensemble_rows.append(summarize_cv_scores(name, scores))

ensemble_results = (
    pd.DataFrame(ensemble_rows)
    .sort_values(
        ["recall_cv_mean", "f2_cv_mean", "precision_cv_mean"],
        ascending=[False, False, False])
    if ensemble_rows else pd.DataFrame()
)

ensemble_results.to_csv(
    REPORTS_DIR / "ensemble_results.csv",
    index=False
)

/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:971: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/utils/_available_if.py", line 32, in _check
    check_result = self.check(obj)
  File "/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/pipeline.py", line 77, in check
    getattr(self._final_estimator, attr)
AttributeError: 'LinearSVC' object has no attribute 'predict_proba'. Did you mean: '_predict_proba_lr'?

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/alexandralozano/miniforge3/envs/dp261-g1/lib/python3.10/site-packages/sklearn/metrics/_scorer.py", line 152, in __call__
    score = scorer._score(
  File "/Users

In [16]:
# Ver y guardar resultados finales

models_results_path = REPORTS_DIR / "models_results.csv"
best_thresholds_path = REPORTS_DIR / "models_best_thresholds.csv"

display(final_results[["source"] + cols_to_show])

threshold_cols = [
    c for c in [
        "source", "model", "threshold",
        "recall", "f2", "precision",
        "positive_rate", "tp", "fp", "fn", "tn"
    ]
    if c in final_best_thresholds.columns
]

display(final_best_thresholds[threshold_cols])

final_results.to_csv(models_results_path, index=False)
final_best_thresholds.to_csv(best_thresholds_path, index=False)

print("Guardado:", models_results_path)
print("Guardado:", best_thresholds_path)

,source,model,recall_cv_mean,f2_cv_mean,precision_cv_mean,f1_cv_mean,f05_cv_mean,roc_auc_cv_mean,average_precision_cv_mean,fit_time_mean
0,tuned_loaded,DecisionTree,0.613821,0.456661,0.228596,0.331660,0.260947,0.718980,0.421301,0.485702
1,tuned_loaded,LogisticRegression,0.610976,0.483195,0.263126,0.367824,0.296933,0.754036,0.436023,0.378450
2,tuned_loaded,LinearSVM,0.603252,0.481969,0.267182,0.370319,0.300677,0.754027,0.430728,0.423806
3,advanced_boosting,XGBoost,0.582520,0.482343,0.285934,0.383508,0.318324,0.766292,0.477231,0.309506
4,advanced_boosting,LightGBM,0.536585,0.468746,0.311559,0.394115,0.340039,0.766080,0.479273,0.404001
5,bagging,Bagging_DecisionTree,0.532520,0.465843,0.310670,0.392293,0.338860,0.756702,0.468173,6.370873
6,ensemble,Stacking_lr,0.232927,0.271256,0.795794,0.360189,0.536109,0.756104,0.450171,4.170145
7,ensemble,Voting_soft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.242831


,source,model,threshold,recall,f2,precision,positive_rate,tp,fp,fn,tn
0,tuned_loaded,LogisticRegression,0.02,1.00000,0.412489,0.123129,0.99895,2460,17519,0,21
1,tuned_loaded,LinearSVM,0.69,1.00000,0.412323,0.123055,0.99955,2460,17531,0,9
2,tuned_loaded,DecisionTree,0.01,0.94878,0.420192,0.130151,0.89665,2334,15599,126,1941


Guardado: /Users/alexandralozano/dp261-g1/reports/models_results.csv
Guardado: /Users/alexandralozano/dp261-g1/reports/models_best_thresholds.csv


Se seleccionó LogisticRegression tuneado con threshold 0.50 como modelo final porque ofrece el mejor equilibrio entre detección de Bad Buys y viabilidad comercial. Aunque DecisionTree obtuvo un recall ligeramente mayor, la diferencia fue mínima. LogisticRegression presentó mejor F2 y mayor precision, lo que indica que logra detectar una proporción relevante de compras malas sin castigar excesivamente las compras buenas. 

Además, se evaluó threshold tuning y se observó que umbrales muy bajos, como 0.02, elevan el recall hasta casi 1.00, pero reducen mucho la precision. En términos de negocio, esto significa que el modelo marcaría demasiados autos como riesgosos, incluyendo muchos autos buenos. Esto podría reducir demasiado el inventario disponible y afectar directamente las ventas. Por ello, se decidió mantener el threshold 0.50 como punto operativo inicial. Este umbral permite capturar una parte importante de los Bad Buys, manteniendo un nivel más razonable de falsos positivos. En otras palabras, el modelo no solo busca minimizar pérdidas por comprar autos malos, sino también evitar bloquear oportunidades comerciales rentables.

En conclusión, LogisticRegression tuneado con threshold 0.50 representa una solución más balanceada, interpretable y operativamente viable para apoyar la decisión de compra.